In [ ]:
import networkx as nx
from collections import defaultdict
import numpy as np



In [42]:
#read data from Nextstrain's traits.json file
#models -> county -> alphabet, models -> county -> transition_matrix  (NOTE there is also rate and equibrium_probabilities) 
def read_traits_json(file_path):
    import json
    with open(file_path, 'r') as f:
        data = json.load(f)
    
    models = data['models']
    
    county_names = models['county']['alphabet']
    transition_matrix = models['county']["transition_matrix"]
    return county_names, transition_matrix

def align_and_normalize_matrix(run_matrix, run_alphabet, master_county_names):
    """
    Aligns a sparse transition matrix from Nextstrain to a master list of counties 
    and row-normalizes it so rows sum to 1.
    """
    N = len(master_county_names)
    full_mat = np.zeros((N, N))
    
    # Create a mapping from the run's alphabet index to the master list's index
    index_map = []
    for county in run_alphabet:
        if county in master_county_names:
            index_map.append(master_county_names.index(county))
        else:
            index_map.append(-1) # Ignore unexpected counties
            
    # Map the sparse matrix values to their exact, absolute coordinates in the full matrix
    for i in range(len(run_alphabet)):
        master_i = index_map[i]
        if master_i == -1: 
            continue
            
        for j in range(len(run_alphabet)):
            master_j = index_map[j]
            if master_j == -1: 
                continue
                
            full_mat[master_i, master_j] = run_matrix[i][j]
            
    # Row-normalize the full matrix
    row_sums = full_mat.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0  # avoid division by zero for rows with no outbound transfers
    normalized = full_mat / row_sums
    
    return normalized

In [ ]:
county_names, epihiper_matrix = read_traits_json('/Users/anwarren/projects/git_repos/ndssl/BeyondBaseline/Data/test_mugration/linelist_mugration.json')
#LASSO greedy 50 subgroups, 1s(1-LL)
county_names_LASSOGreedy, linelist_matrix_LASSOGreedy = read_traits_json('/Users/anwarren/projects/git_repos/ndssl/BeyondBaseline/Data/test_mugration/run_03_vadelta_2026_04_15_128to428_LASSOGreedy_traits.json')
#1S SURS
county_names_SURS, linelist_matrix_SURS = read_traits_json('/Users/anwarren/projects/git_repos/ndssl/BeyondBaseline/Data/test_mugration/run_03_vadelta_2026_03_22_128to428_SURS_traits.json')

In [43]:
#normalize the matrices
normalized_epihiper_matrix = align_and_normalize_matrix(epihiper_matrix, county_names, county_names)
normalized_lassogreedy_matrix = align_and_normalize_matrix(linelist_matrix_LASSOGreedy, county_names_LASSOGreedy, county_names)
normalized_SURS_matrix = align_and_normalize_matrix(linelist_matrix_SURS, county_names_SURS, county_names)

In [44]:
#/Users/anwarren/projects/git_repos/ndssl/BeyondBaseline/Data/test_mugration/run_03_vadelta_2026_03_22_128to428_SURS_traits.json
normalized_SURS_matrix

array([[0.        , 0.00755012, 0.00755137, ..., 0.00755338, 0.00755227,
        0.00755642],
       [0.00715621, 0.        , 0.00857373, ..., 0.00715811, 0.00715757,
        0.00737589],
       [0.00504839, 0.00604739, 0.        , ..., 0.0050491 , 0.00504923,
        0.00578879],
       ...,
       [0.00690663, 0.00690549, 0.00690577, ..., 0.        , 0.00924673,
        0.00690909],
       [0.00697021, 0.00696955, 0.00697054, ..., 0.00933323, 0.        ,
        0.0069732 ],
       [0.00527002, 0.00542727, 0.00603889, ..., 0.00526977, 0.00526938,
        0.        ]], shape=(134, 134))

In [37]:
#print names that don't match between county_names and county_names_LASSOGreedy
missing =[]
for i,name in enumerate(county_names):
    if name not in county_names_SURS:
        missing.append(name)
print(f"Missing counties in SURS: {len(missing)}")

Missing counties in SURS: 1


In [23]:
#print names that don't match between county_names and county_names_LASSOGreedy
for i,name in enumerate(county_names):
    if name != county_names_LASSOGreedy[i]:
        print(f"{name} not in LASSOGreedy county names")

In [25]:
normal_lassogreedy_matrix = normalize_transition_matrix(linelist_matrix_LASSOGreedy)

In [47]:
county_names

['',
 'Accomack VA',
 'Albemarle VA',
 'Alexandria City VA',
 'Alleghany VA',
 'Amelia VA',
 'Amherst VA',
 'Appomattox VA',
 'Arlington VA',
 'Augusta VA',
 'Bath VA',
 'Bedford VA',
 'Bland VA',
 'Botetourt VA',
 'Bristol City VA',
 'Brunswick VA',
 'Buchanan VA',
 'Buckingham VA',
 'Buena Vista City VA',
 'Campbell VA',
 'Caroline VA',
 'Carroll VA',
 'Charles City VA',
 'Charlotte VA',
 'Charlottesville City VA',
 'Chesapeake City VA',
 'Chesterfield VA',
 'Clarke VA',
 'Colonial Heights City VA',
 'Covington City VA',
 'Craig VA',
 'Culpeper VA',
 'Cumberland VA',
 'Danville City VA',
 'Dickenson VA',
 'Dinwiddie VA',
 'Emporia City VA',
 'Essex VA',
 'Fairfax City VA',
 'Fairfax VA',
 'Falls Church City VA',
 'Fauquier VA',
 'Floyd VA',
 'Fluvanna VA',
 'Franklin City VA',
 'Franklin VA',
 'Frederick VA',
 'Fredericksburg City VA',
 'Galax City VA',
 'Giles VA',
 'Gloucester VA',
 'Goochland VA',
 'Grayson VA',
 'Greene VA',
 'Greensville VA',
 'Halifax VA',
 'Hampton City VA',
 

In [48]:
import numpy as np
from scipy.stats import pearsonr

def get_off_diagonals(matrix, alphabet):
    """Flattens a matrix into a 1D array, ignoring the diagonal."""
    N = len(alphabet)
    off_diags = []
    if county_names[0] =="" or county_names[0] == "?":
        initial_index = 1
    else:
        initial_index = 0
    
    for i in range(initial_index, N):
        for j in range(initial_index, N):
            if i!= j:
                off_diags.append(matrix[i][j])
    return np.array(off_diags)

# Assume abm_matrix, nextstrain_run1_matrix, and nextstrain_run2_matrix 
# are already row-normalized and share the exact same 'alphabet' order.

abm_flat = get_off_diagonals(normalized_epihiper_matrix, county_names)
run1_flat = get_off_diagonals(normalized_lassogreedy_matrix, county_names)
run2_flat = get_off_diagonals(normalized_SURS_matrix, county_names)

# 1. Pearson Correlation
r_1, _ = pearsonr(abm_flat, run1_flat)
r_2, _ = pearsonr(abm_flat, run2_flat)

print(f"Run 1 Pearson Correlation: {r_1:.4f}")
print(f"Run 2 Pearson Correlation: {r_2:.4f}")

# 2. Mean Absolute Error (L1 Distance)
mae_1 = np.mean(np.abs(abm_flat - run1_flat))
mae_2 = np.mean(np.abs(abm_flat - run2_flat))

print(f"Run 1 MAE: {mae_1:.4f}")
print(f"Run 2 MAE: {mae_2:.4f}")

Run 1 Pearson Correlation: 0.4494
Run 2 Pearson Correlation: 0.4807
Run 1 MAE: 0.0112
Run 2 MAE: 0.0111
